# Phase 1: Python Foundations (Core Mechanics for AI Engineering)
Welcome to **Phase 1** of the Python for AI Developer Curriculum!
Before touching neural networks, LLMs, or massive datasets, you must build deep fluency in how Python stores data in memory, controls execution flow, and structures functions.

### Topics Covered in This Notebook:
1. **Environment & Tooling:** Python runtime, virtual environments (`venv`, `conda`, `uv`), package management (`pip`, `poetry`), VS Code + WSL setup, and `__name__ == "__main__"`.
2. **Syntax & Primitives:** Variables, dynamic typing, object references, primitives (`int`, `float`, `str`, `bool`), type casting, and string manipulation (indexing, slicing, `f-strings`).
3. **Operators & Expressions:** Arithmetic, comparison, logical (`and`, `or`, `not`), membership (`in`), identity (`is` vs `==`), and bitwise operators (`&`, `|`, `^`, `~`, `<<`, `>>`).
4. **Control Flow:** Conditionals (`if`/`elif`/`else`, `match-case`), loops (`for`, `while`), and loop helpers (`break`, `continue`, `pass`, `enumerate`, `zip`).
5. **Core Data Structures:**
   - **Lists:** Dynamic arrays, slicing, sorting, and mutability pitfalls.
   - **Tuples:** Immutable sequences, packing/unpacking, tensor shape representations.
   - **Dictionaries:** Hash maps, $O(1)$ lookups, `.get()`, `.setdefault()`, nested JSON/LLM payloads.
   - **Sets:** Unique collections and set math (`|`, `&`, `-`, `^`).
6. **Functions & Scope:** `def`, positional vs. keyword arguments, mutable default parameter traps, `*args` and `**kwargs`, `lambda` functions, and the **LEGB** scope rule.

---
## 1. Environment & Tooling
In professional AI development, isolating project dependencies is critical because libraries like `torch`, `transformers`, and `pydantic` evolve rapidly and often have strict version constraints.

### Essential CLI Commands Reference (Run in Terminal / WSL Bash):
```bash
# 1. Standard venv (Built-in)
python3 -m venv .venv
source .venv/bin/activate        # Linux / WSL / macOS
# .venv\Scripts\activate       # Windows PowerShell

# 2. uv (Modern, ultra-fast Rust-based Python package & env manager - Highly Recommended for AI)
uv venv
source .venv/bin/activate
uv pip install numpy pandas torch

# 3. Conda (Popular when managing non-Python CUDA/C++ binaries)
conda create -n ai_env python=3.11 -y
conda activate ai_env

# 4. Pip & Requirements Locking
pip install -r requirements.txt
pip freeze > requirements.txt
```

Let's inspect our current Python runtime environment from inside the notebook:

In [ ]:
import sys
import platform

print(f"Python Version      : {sys.version.split()[0]}")
print(f"Executable Path     : {sys.executable}")
print(f"Operating System    : {platform.system()} ({platform.release()})")
print(f"Top-Level Module    : {__name__}")

# Demonstration of the Main Guard pattern used in production training scripts:
def main():
    print("Executing main entrypoint logic (e.g., starting training pipeline)...")

if __name__ == "__main__":
    main()

---
## 2. Syntax & Primitives
In Python, **everything is an object**, and variables are simply **labels (references)** pointing to objects in memory, rather than fixed-size memory boxes.
* **Dynamic Typing:** Type is attached to the *object*, not the variable name.
* **Immutability of Primitives:** `int`, `float`, `bool`, and `str` are **immutable**. Modifying them creates a brand-new object at a new memory address (`id()`).

In [ ]:
# 1. Variables, Dynamic Typing & Memory Addresses
learning_rate = 0.001          # float
epochs = 50                    # int
model_name = "Llama-3-8B"      # str
use_fp16 = True                # bool

print(f"learning_rate: {learning_rate} ({type(learning_rate).__name__}) | Memory ID: {id(learning_rate)}")

# Rebinding an immutable int creates a new object in memory
initial_id = id(epochs)
epochs += 10
print(f"epochs before vs after ID same? {initial_id == id(epochs)} (Old: {initial_id}, New: {id(epochs)})")

# 2. Type Casting (Explicit Conversion)
raw_batch_size = "64"
raw_dropout = "0.25"
batch_size = int(raw_batch_size)
dropout = float(raw_dropout)
print(f"Converted batch_size: {batch_size * 2} | dropout: {dropout}")

# Truthy and Falsy values in Python:
# Falsy: 0, 0.0, "", [], (), {}, set(), None, False
for val in [0, 1, "", "prompt", [], [10]]:
    print(f"Value: {repr(val):<10} -> bool: {bool(val)}")

In [ ]:
# 3. String Manipulation: Indexing, Slicing [start:stop:step], and f-strings
prompt = "  System: You are a helpful AI assistant. User: Explain backpropagation.  "

# Cleaning and normalizing strings (crucial for NLP & RAG preprocessing)
cleaned = prompt.strip()
print("Cleaned      :", cleaned)
print("Lowercase    :", cleaned.lower()[:38] + "...")
print("Starts with  :", cleaned.startswith("System:"))
print("Replaced     :", cleaned.replace("helpful AI", "expert ML"))

# Slicing [start:stop:step]
token_stream = "ABCDEFGHIJ"
print("First 4 chars:", token_stream[:4])     # 'ABCD'
print("Last 3 chars :", token_stream[-3:])    # 'HIJ'
print("Every 2nd    :", token_stream[::2])    # 'ACEGI'
print("Reversed     :", token_stream[::-1])   # 'JIHGFEDCBA'

# Advanced f-string formatting (for clean training logs)
epoch = 7
loss = 0.03456789
accuracy = 0.9428
print(f"[Epoch {epoch:03d}] Loss: {loss:.4f} | Accuracy: {accuracy:.2%} | Scientific LR: {learning_rate:.1e}")

---
## 3. Operators & Expressions
Understanding how Python evaluates expressions—especially **identity (`is`) vs. equality (`==`)**, **short-circuit logical evaluation**, and **bitwise operators**—is vital when writing data masks, quantization scripts, and permission flags.

In [ ]:
# 1. Arithmetic Operators
total_samples = 10_050
batch_size = 64

full_batches = total_samples // batch_size     # Floor division
leftover_samples = total_samples % batch_size  # Modulo (remainder)
l2_penalty = 0.01 ** 2                         # Exponentiation
print(f"Total: {total_samples} -> {full_batches} full batches + {leftover_samples} leftover samples")

# Using divmod() to get quotient and remainder in one step:
b, rem = divmod(total_samples, batch_size)
print(f"divmod result: batches={b}, remainder={rem}")

# 2. Identity (is) vs Equality (==)
# '==' checks if two objects have the same VALUE.
# 'is' checks if two variables point to the EXACT SAME OBJECT in memory (id(a) == id(b)).
vec_a = [1.0, 2.0, 3.0]
vec_b = [1.0, 2.0, 3.0]
vec_c = vec_a

print(f"vec_a == vec_b (Same Value?)   : {vec_a == vec_b}")
print(f"vec_a is vec_b (Same Memory?)  : {vec_a is vec_b}")
print(f"vec_a is vec_c (Same Reference): {vec_a is vec_c}")

# Rule of thumb: ALWAYS use 'is None' or 'is not None' when checking for None
api_key = None
if api_key is None:
    print("API key is not configured yet.")

In [ ]:
# 3. Logical Operators (Short-Circuit Evaluation) & Membership (in)
supported_models = {"gpt-4o", "claude-3-5-sonnet", "gemini-2.5-pro", "llama-3.1-70b"}
requested_model = "gemini-2.5-pro"

if requested_model in supported_models:
    print(f"Model '{requested_model}' is supported! (O(1) set membership lookup)")

# Short-circuiting default assignment pattern:
user_custom_prompt = ""
active_prompt = user_custom_prompt or "Default System Prompt: Be concise."
print("Active Prompt:", active_prompt)

# 4. Bitwise Operators (&, |, ^, ~, <<, >>)
# Widely used in binary masks, image processing, permission flags, and Pandas/NumPy array filtering!
FLAG_READ    = 0b001  # 1
FLAG_WRITE   = 0b010  # 2
FLAG_EXECUTE = 0b100  # 4

# Combine permissions using Bitwise OR (|)
user_permissions = FLAG_READ | FLAG_WRITE  # 0b011 (3)
print(f"User Permissions Binary: {bin(user_permissions)} (Decimal: {user_permissions})")

# Check if a specific bit flag is set using Bitwise AND (&)
can_write = bool(user_permissions & FLAG_WRITE)
can_exec  = bool(user_permissions & FLAG_EXECUTE)
print(f"Can Write? {can_write} | Can Execute? {can_exec}")

# Toggle a flag using Bitwise XOR (^) and Bit Shifts (<<, >>)
user_permissions ^= FLAG_WRITE  # Removes WRITE flag
print(f"After XOR toggle WRITE : {bin(user_permissions)}")
print(f"1 << 8 (2^8 fast power): {1 << 8} | 64 >> 2 (Divide by 4): {64 >> 2}")

---
## 4. Control Flow
AI training loops, early-stopping logic, and agent routing rely on clean control flow. Python 3.10+ also introduced structural pattern matching (`match-case`), which is awesome for routing LLM tool calls or parsing structured API responses.

In [ ]:
# 1. Conditionals (if / elif / else) & Ternary Expression
val_loss = 0.18
best_loss = 0.20

if val_loss < best_loss:
    best_loss = val_loss
    status = "Saved new checkpoint"
elif val_loss == best_loss:
    status = "Loss plateaued"
else:
    status = "Loss increased (potential overfitting)"

# Inline ternary operator
device = "cuda" if True else "cpu"
print(f"Status: {status} | Running on device: {device}")

# 2. Structural Pattern Matching (match-case) for Agent Tool Routing
tool_call = {"tool": "vector_search", "query": "transformer attention", "top_k": 5}

match tool_call:
    case {"tool": "vector_search", "query": str(q), "top_k": int(k)}:
        print(f"[Router] Executing Vector Search for '{q}' (top_k={k})")
    case {"tool": "sql_query", "sql": str(sql)}:
        print(f"[Router] Executing SQL: {sql}")
    case _:
        print("[Router] Unknown tool call schema!")

In [ ]:
# 3. Loops (for, while), break, continue, enumerate(), and zip()
# Simulating a mini training loop with Early Stopping and NaN skipping
epoch_losses = [0.85, 0.62, float("nan"), 0.41, 0.19, 0.185, 0.184]
target_loss_threshold = 0.20

for epoch_idx, current_loss in enumerate(epoch_losses, start=1):
    # Skip corrupted NaN batches using 'continue'
    if current_loss != current_loss:  # NaN is the only float not equal to itself!
        print(f"Epoch {epoch_idx}: [WARNING] NaN loss detected! Skipping update...")
        continue

    print(f"Epoch {epoch_idx}: Training loss = {current_loss:.3f}")

    # Stop training early once target loss is reached using 'break'
    if current_loss <= target_loss_threshold:
        print(f"--> Early stopping triggered at Epoch {epoch_idx}! Target {target_loss_threshold} reached.")
        break

# Pairing inputs, predictions, and ground-truth labels cleanly with zip()
prompts = ["2+2=", "Capital of France?", "Python creator?"]
preds   = ["4", "Paris", "Guido van Rossum"]
scores  = [0.99, 0.98, 0.95]

print("\n--- Batch Evaluation with zip() ---")
for prompt, pred, conf in zip(prompts, preds, scores):
    print(f"Prompt: {prompt:<20} -> Pred: {pred:<18} (Confidence: {conf:.2f})")

---
## 5. Core Data Structures
Choosing the right data structure directly impacts the speed and memory footprint of your AI pipelines:
* **`list`**: Mutable, ordered sequence. Great for collecting batch outputs ($O(1)$ append, $O(N)$ membership lookup).
* **`tuple`**: Immutable, ordered sequence. Used for fixed shapes (e.g., `(batch_size, channels, height, width)`), dictionary keys, and returning multiple values from functions.
* **`dict`**: Mutable hash map of `key: value` pairs. $O(1)$ average lookup time. The backbone of JSON API responses and model configurations.
* **`set`**: Mutable unordered collection of unique hashable elements. $O(1)$ membership check (`x in my_set`) and instant set math (vocabulary overlap, stopword removal).

In [ ]:
# 1. Lists: Dynamic Arrays, Sorting & The Mutability Aliasing Trap
layer_sizes = [128, 256, 512]
layer_sizes.append(1024)
layer_sizes.extend([2048, 4096])
popped = layer_sizes.pop()  # Removes and returns 4096
print("Layer sizes:", layer_sizes, "| Popped:", popped)

# Custom sorting with a key function (e.g., sorting model candidates by validation F1 score)
candidates = [
    {"model": "RandomForest", "f1": 0.84},
    {"model": "XGBoost", "f1": 0.91},
    {"model": "LogisticRegression", "f1": 0.79},
]
ranked = sorted(candidates, key=lambda item: item["f1"], reverse=True)
print("Ranked Models:", ranked)

# CRITICAL TRAP: List Aliasing vs Copying
original_weights = [0.1, 0.2, 0.3]
bad_alias = original_weights          # Points to the SAME list in memory!
safe_copy = original_weights.copy()   # Creates a new shallow copy (or use original_weights[:])

bad_alias[0] = 999.0
print(f"Original after modifying alias: {original_weights} | Safe copy: {safe_copy}")

In [ ]:
# 2. Tuples: Immutable Sequences, Tensor Shapes & Unpacking
image_tensor_shape = (32, 3, 224, 224)  # (Batch, Channels, Height, Width)

# Extended Iterable Unpacking with *
batch, channels, *spatial_dims = image_tensor_shape
print(f"Batch: {batch}, Channels: {channels}, Spatial Dims: {spatial_dims}")

# Swapping variables in-place using tuple unpacking
a, b = "Model_v1", "Model_v2"
a, b = b, a
print(f"Swapped: a={a}, b={b}")

# 3. Dictionaries: Hash Maps, Safe Lookups & Nested LLM Payloads
llm_response = {
    "id": "chatcmpl-987",
    "model": "gpt-4o-mini",
    "usage": {"prompt_tokens": 120, "completion_tokens": 45},
    "choices": [{"message": {"role": "assistant", "content": "Gradient descent minimizes loss."}}]
}

# Safe retrieval with .get() avoids KeyError in production pipelines
finish_reason = llm_response.get("finish_reason", "stop")
content = llm_response.get("choices", [{}])[0].get("message", {}).get("content", "")
print(f"Extracted Content: '{content}' | Default Finish Reason: '{finish_reason}'")

# Iterating over keys and values + dictionary merging (| operator in Python 3.9+)
base_config = {"lr": 1e-3, "epochs": 10, "optimizer": "AdamW"}
override_config = {"epochs": 25, "weight_decay": 0.01}
merged_config = base_config | override_config
print("Merged Training Config:", merged_config)

In [ ]:
# 4. Sets: Vocabulary Deduplication & Set Math
doc_1_tokens = {"transformer", "attention", "matrix", "embeddings", "python", "attention"}
doc_2_tokens = {"python", "pytorch", "matrix", "gpu", "tensors"}

print("Unique tokens in Doc 1 :", doc_1_tokens)
print("Intersection (&)       :", doc_1_tokens & doc_2_tokens)  # Shared vocabulary
print("Union (|)              :", doc_1_tokens | doc_2_tokens)  # Combined vocabulary
print("Difference (-)         :", doc_1_tokens - doc_2_tokens)  # In Doc 1 but NOT Doc 2
print("Symmetric Diff (^)     :", doc_1_tokens ^ doc_2_tokens)  # In either, but not both

# Computing Jaccard Similarity between two documents (a classic NLP retrieval metric!)
jaccard_sim = len(doc_1_tokens & doc_2_tokens) / len(doc_1_tokens | doc_2_tokens)
print(f"Jaccard Similarity     : {jaccard_sim:.3f}")

---
## 6. Functions & Scope
In AI libraries like PyTorch and Hugging Face, functions and class constructors frequently use `*args` and `**kwargs` to forward flexible configurations across layers. You must also understand the **Mutable Default Argument Trap** and the **LEGB (Local, Enclosing, Global, Built-in)** scope resolution order.

In [ ]:
# 1. Positional, Keyword, Default Parameters, *args, and **kwargs
def run_inference(prompt: str, temperature: float = 0.7, *stop_sequences: str, **model_kwargs) -> dict:
    """Simulates calling an LLM API with flexible arguments."""
    return {
        "prompt": prompt,
        "temperature": temperature,
        "stop_sequences": stop_sequences,  # Captured as a tuple
        "extra_params": model_kwargs       # Captured as a dict
    }

result = run_inference(
    "Explain LoRA fine-tuning",
    0.2,
    "\nUser:", "<|endoftext|>",
    top_p=0.95,
    max_tokens=512,
    stream=True
)
print("Inference Call Result:", result)

# 2. CRITICAL INTERVIEW & PRODUCTION TRAP: Mutable Default Arguments
# NEVER do: def add_token(token, history=[]): -> The default list is created ONCE at function definition time!
def append_message_safe(role: str, content: str, chat_history: list | None = None) -> list:
    if chat_history is None:
        chat_history = []  # Creates a fresh list on every call when not provided
    chat_history.append({"role": role, "content": content})
    return chat_history

session_1 = append_message_safe("user", "Hello AI")
session_2 = append_message_safe("user", "Fresh session")
print("Session 1:", session_1)
print("Session 2 (Isolated!):", session_2)

In [ ]:
# 3. Lambda Functions & The LEGB Scope Rule (Local -> Enclosing -> Global -> Built-in)
GLOBAL_SEED = 42  # Global Scope (G)

def create_lr_scheduler(initial_lr: float):
    # 'initial_lr' and 'step_count' live in the Enclosing Scope (E)
    step_count = 0

    def step(decay_factor: float = 0.95) -> float:
        # 'decay_factor' lives in the Local Scope (L)
        nonlocal step_count  # Allows modifying the Enclosing variable
        step_count += 1
        current_lr = initial_lr * (decay_factor ** step_count)
        return round(current_lr, 6)

    return step

scheduler = create_lr_scheduler(0.01)
print("Step 1 LR:", scheduler())
print("Step 2 LR:", scheduler())
print("Step 3 LR:", scheduler())

# Lambda functions for concise inline transformations (e.g., activation functions)
relu = lambda x: max(0.0, x)  # max() is resolved from Built-in Scope (B)
sample_activations = [-2.5, -0.1, 0.0, 1.4, 3.8]
print("ReLU outputs:", [relu(val) for val in sample_activations])